In [1]:
# Cell 1 — Safe install (run once)
# Installs facenet-pytorch but DOES NOT upgrade torch/torchvision (avoids version mismatches).
!pip install -q facenet-pytorch --no-deps


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 37.4 MB/s eta 0:00:00


In [10]:
# Cell 2 — Imports, device, models, and robust query embedding creation
import os
from pathlib import Path
from PIL import Image, ImageEnhance
import numpy as np
import cv2
import torch
import math
import sys

# Attempt to import facenet-pytorch
try:
    from facenet_pytorch import MTCNN, InceptionResnetV1
except Exception as e:
    raise RuntimeError("facenet-pytorch import failed. Ensure you ran the install cell: !pip install -q facenet-pytorch --no-deps") from e

# File paths (change if you prefer)
INPUT_DIR = Path("/content/input")
QUERY_PATH = INPUT_DIR / "presenter_image.png"
if not QUERY_PATH.exists():
    alt = INPUT_DIR / "presenter_image.jpg"
    if alt.exists():
        QUERY_PATH = alt

VIDEO_PATH = INPUT_DIR / "test.mp4"

if not QUERY_PATH.exists():
    raise FileNotFoundError(f"Query image not found. Place query.png or query.jpg in {INPUT_DIR}/")
if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"Video file not found. Place test.mp4 in {INPUT_DIR}/")

# Device and models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

mtcnn = MTCNN(image_size=160, margin=0, keep_all=True, device=device, thresholds=[0.6,0.7,0.7])
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# Helper functions
def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def l2dist(a, b):
    return float(np.linalg.norm(a - b))

# Build multiple query embeddings via simple augmentations (flip, brightness/contrast)
img = Image.open(str(QUERY_PATH)).convert("RGB")
print("Query image size:", img.size)

aug_images = [img, img.transpose(Image.FLIP_LEFT_RIGHT)]
# slight brightness/contrast variants
enh_b = ImageEnhance.Brightness(img)
aug_images += [enh_b.enhance(0.85), enh_b.enhance(1.15)]
enh_c = ImageEnhance.Contrast(img)
aug_images += [enh_c.enhance(0.9), enh_c.enhance(1.1)]

query_embs = []
for aimg in aug_images:
    faces = mtcnn(aimg)  # None or tensor (3,160,160) or (N,3,160,160)
    if faces is None:
        continue
    if faces.ndim == 3:
        faces = faces.unsqueeze(0)
    faces = faces.to(device)
    with torch.no_grad():
        embs = resnet(faces).cpu().numpy()
    # add all detected faces from this augmentation
    for e in embs:
        query_embs.append(e)

if len(query_embs) == 0:
    raise ValueError("No face crops detected in the query image (try a clearer frontal face).")

query_embs = np.stack(query_embs, axis=0)  # (K,512)
print(f"Collected {query_embs.shape[0]} query embedding(s).")


Using device: cuda
Query image size: (321, 292)
Collected 6 query embedding(s).


In [11]:
# Cell 3 — Video metadata and detection attempts (auto-retry)
VIDEO_PATH_STR = str(VIDEO_PATH)
cap = cv2.VideoCapture(VIDEO_PATH_STR)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH_STR}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
duration = (frame_count / fps) if frame_count > 0 else 0.0
print(f"Video FPS: {fps:.3f}, Frames: {frame_count}, Duration: {duration:.2f}s")
cap.release()

# Attempt settings: list of (SAMPLE_SEC, SIM_THRESHOLD)
# It will try them in order until intervals are found.
ATTEMPTS = [
    (15, 0.70),  # default/fast (recommended)
    (10, 0.65),  # denser + looser threshold if first fails
    (5,  0.60),  # densest, loosest — fallback
]

MIN_SEG_LEN = 60.0  # seconds: require at least 1 minute continuous presence
MAX_GAP = 60.0      # seconds: merge gaps <= this

best_intervals = []
attempt_used = None

# core detection function (single attempt)
def run_detection(sample_sec, sim_threshold):
    frame_interval = max(1, int(round(fps * sample_sec)))
    timestamps = []
    cap = cv2.VideoCapture(VIDEO_PATH_STR)
    total_frames = frame_count if frame_count>0 else int(duration*fps)
    for fidx in range(0, total_frames, frame_interval):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
        ret, frame = cap.read()
        if not ret:
            continue
        pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        try:
            faces = mtcnn(pil)
        except Exception:
            continue
        if faces is None:
            continue
        if faces.ndim == 3:
            faces = faces.unsqueeze(0)
        faces = faces.to(device)
        with torch.no_grad():
            embs = resnet(faces).cpu().numpy()   # (N,512)
        # compute best similarity among faces and query_embs
        best_cos = -1.0
        for emb in embs:
            # check against all query embeddings and take best
            for q in query_embs:
                c = cosine_sim(q, emb)
                if c > best_cos:
                    best_cos = c
        if best_cos >= sim_threshold:
            timestamps.append(fidx / fps)
    cap.release()
    # group timestamps into intervals
    intervals = []
    if timestamps:
        timestamps = sorted(timestamps)
        start = timestamps[0]
        last = timestamps[0]
        for t in timestamps[1:]:
            if t - last <= MAX_GAP:
                last = t
            else:
                if last - start >= MIN_SEG_LEN:
                    intervals.append((max(0.0, start), min(duration if duration>0 else last + sample_sec, last + sample_sec)))
                start = last = t
        if last - start >= MIN_SEG_LEN:
            intervals.append((max(0.0, start), min(duration if duration>0 else last + sample_sec, last + sample_sec)))
    return intervals, timestamps

# run attempts
for sample_sec, sim_t in ATTEMPTS:
    print(f"\nAttempt: sample={sample_sec}s, sim_threshold={sim_t} ...")
    intervals, raw_ts = run_detection(sample_sec, sim_t)
    print("  raw detections (sample points):", raw_ts[:10], "..." if len(raw_ts)>10 else "")
    if intervals:
        best_intervals = intervals
        attempt_used = (sample_sec, sim_t)
        print("  --> Intervals FOUND on this attempt.")
        break
    else:
        print("  --> No intervals found on this attempt. Retrying with next settings...")

# final outcome
if not best_intervals:
    print("\nNo presenter intervals detected after all attempts.")
else:
    print(f"\nPresenter intervals detected (using sample={attempt_used[0]}s, threshold={attempt_used[1]}):")
    for s,e in best_intervals:
        print(f" - Start: {s:.2f}s, End: {e:.2f}s")


Video FPS: 29.970, Frames: 21194, Duration: 707.17s

Attempt: sample=15s, sim_threshold=0.7 ...
  raw detections (sample points): [15.015, 45.045, 135.135, 150.15, 225.225, 240.24, 255.255, 270.27] 
  --> No intervals found on this attempt. Retrying with next settings...

Attempt: sample=10s, sim_threshold=0.65 ...
  raw detections (sample points): [10.01, 20.02, 40.04, 50.050000000000004, 60.06, 70.07000000000001, 140.14000000000001, 150.15, 220.22, 240.24] ...
  --> Intervals FOUND on this attempt.

Presenter intervals detected (using sample=10s, threshold=0.65):
 - Start: 10.01s, End: 80.07s
 - Start: 220.22s, End: 290.28s


In [12]:
# Cell 4 — Save results to text file and print
OUT_PATH = "/content/output_timestamps.txt"
with open(OUT_PATH, "w") as fh:
    if not best_intervals:
        fh.write("No presenter intervals detected.\n")
    else:
        for s,e in best_intervals:
            fh.write(f"Start: {s:.2f} sec, End: {e:.2f} sec\n")

print("\nSaved output to:", OUT_PATH)
with open(OUT_PATH, "r") as fh:
    print(fh.read())



Saved output to: /content/output_timestamps.txt
Start: 10.01 sec, End: 80.07 sec
Start: 220.22 sec, End: 290.28 sec



In [13]:
# ===== Refine start times for detected intervals (run after main pipeline) =====
import os, math, cv2, numpy as np
from pathlib import Path
from PIL import Image
import torch

# PARAMETERS - tune if needed
REFINE_WINDOW = 20.0        # how many seconds back to search from coarse start
STEP_SEC = 0.5              # step size in seconds for refinement (0.5s is precise)
REFINE_SIM_THRESHOLD = 0.68 # slightly looser than main threshold; adjust if needed

# Paths used by pipeline (adjust if you changed them)
INPUT_DIR = Path("/content/input")
VIDEO_PATH = str(INPUT_DIR / "test.mp4")
QUERY_PATH = INPUT_DIR / "query.png"
if not Path(QUERY_PATH).exists():
    alt = INPUT_DIR / "query.jpg"
    if alt.exists():
        QUERY_PATH = str(alt)

# Ensure we have coarse intervals from previous run
# Try to use best_intervals variable from memory; else read the output file
try:
    best_intervals  # noqa: F821
except NameError:
    # try to read from output file that pipeline wrote
    out_file = Path("/content/output_timestamps.txt")
    best_intervals = []
    if out_file.exists():
        for line in out_file.read_text().splitlines():
            if line.strip() and "Start" in line and "End" in line:
                parts = line.replace(",", "").replace("Start:", "").replace("End:", "").split()
                try:
                    s = float(parts[0])
                    e = float(parts[1])
                    best_intervals.append((s, e))
                except:
                    pass
    if not best_intervals:
        raise RuntimeError("No coarse intervals found in memory or /content/output_timestamps.txt. Run the pipeline first.")

print("Coarse intervals (before refinement):", best_intervals)

# Ensure models and query embeddings exist; try to reuse existing variables otherwise (safe re-init)
need_init = False
try:
    mtcnn, resnet, query_embs, device, fps  # noqa: F821
except NameError:
    need_init = True

if need_init:
    print("Models/embeddings not found in session — initializing facenet models and query embeddings (safe init)...")
    # safe install assumption: facenet-pytorch already installed with --no-deps earlier
    try:
        from facenet_pytorch import MTCNN, InceptionResnetV1
    except Exception as e:
        raise RuntimeError("facenet-pytorch not available. Run: !pip install -q facenet-pytorch --no-deps") from e

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    mtcnn = MTCNN(image_size=160, margin=0, keep_all=True, device=device, thresholds=[0.6,0.7,0.7])
    resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

    # build query_embs (simple single-face extraction here)
    if not Path(QUERY_PATH).exists():
        raise FileNotFoundError(f"Query image not found at {QUERY_PATH}")
    qimg = Image.open(str(QUERY_PATH)).convert("RGB")
    faces = mtcnn(qimg)
    if faces is None:
        raise ValueError("No face detected in the query image during refinement init. Provide clear frontal face.")
    if faces.ndim == 3:
        faces = faces.unsqueeze(0)
    faces = faces.to(device)
    with torch.no_grad():
        q_embs_t = resnet(faces).cpu().numpy()
    query_embs = q_embs_t  # (K,512)
    # load fps
    cap_tmp = cv2.VideoCapture(VIDEO_PATH)
    fps = cap_tmp.get(cv2.CAP_PROP_FPS) or 25.0
    cap_tmp.release()

# helper sim
def cosine(a,b):
    return float(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b)))

# open video for random access
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError("Cannot open video at: " + VIDEO_PATH)

refined_intervals = []
for (coarse_start, coarse_end) in best_intervals:
    # define search window
    search_start = max(0.0, coarse_start - REFINE_WINDOW)
    search_end = coarse_start  # exclusive
    found_earliest = None

    t = search_start
    while t <= search_end + 1e-6:
        frame_idx = int(round(t * fps))
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret:
            t += STEP_SEC
            continue
        pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        try:
            faces = mtcnn(pil)
        except Exception:
            faces = None
        if faces is None:
            t += STEP_SEC
            continue
        if faces.ndim == 3:
            faces = faces.unsqueeze(0)
        faces = faces.to(device)
        with torch.no_grad():
            embs = resnet(faces).cpu().numpy()
        # get best similarity against any query embedding
        best_cos = -1.0
        for emb in embs:
            for q in query_embs:
                c = cosine(q, emb)
                if c > best_cos:
                    best_cos = c
        # check against refine threshold
        if best_cos >= REFINE_SIM_THRESHOLD:
            found_earliest = t
            break
        t += STEP_SEC

    if found_earliest is None:
        # if nothing found in window, keep coarse start
        refined_start = coarse_start
    else:
        refined_start = found_earliest

    # keep end same for now, or optionally extend to include SAMPLE window
    refined_intervals.append((refined_start, coarse_end))

cap.release()

# show before / after and save file
print("Refined intervals (start times corrected):")
for before, after in zip(best_intervals, refined_intervals):
    print(f"Coarse start: {before[0]:.2f}s -> Refined start: {after[0]:.2f}s   (end unchanged: {after[1]:.2f}s)")

out_path = "/content/output_timestamps_refined.txt"
with open(out_path, "w") as fh:
    for s,e in refined_intervals:
        fh.write(f"Start: {s:.2f} sec, End: {e:.2f} sec\n")

print("Saved refined intervals to:", out_path)


Coarse intervals (before refinement): [(10.01, 80.07000000000001), (220.22, 290.28000000000003)]
Refined intervals (start times corrected):
Coarse start: 10.01s -> Refined start: 5.00s   (end unchanged: 80.07s)
Coarse start: 220.22s -> Refined start: 217.72s   (end unchanged: 290.28s)
Saved refined intervals to: /content/output_timestamps_refined.txt


In [14]:
# ===== Merge presenter intervals into one continuous clip =====
from moviepy.editor import VideoFileClip

# Input/Output paths
VIDEO_PATH = "/content/input/test.mp4"
REFINED_PATH = "/content/output_timestamps_refined.txt"
OUTPUT_PATH = "/content/requested_presenter.mp4"

# Parse refined intervals from file
intervals = []
with open(REFINED_PATH, "r") as fh:
    for line in fh:
        if "Start:" in line and "End:" in line:
            parts = line.replace("Start:", "").replace("End:", "").replace("sec", "").split(",")
            s = float(parts[0].strip())
            e = float(parts[1].strip())
            intervals.append((s, e))

if not intervals:
    raise RuntimeError("No intervals found in refined file. Make sure /content/output_timestamps_refined.txt exists and has valid lines.")

# Merge: take earliest start and latest end
global_start = min(s for s, e in intervals)
global_end   = max(e for s, e in intervals)
print(f"Using global segment: Start={global_start:.2f}s, End={global_end:.2f}s")

# Extract subclip
clip = VideoFileClip(VIDEO_PATH).subclip(global_start, global_end)

# Write output (fastest codec for Colab, audio preserved)
clip.write_videofile(OUTPUT_PATH, codec="libx264", audio_codec="aac", threads=4, preset="ultrafast")

print("Saved merged presenter clip to:", OUTPUT_PATH)


Using global segment: Start=5.00s, End=290.28s
Moviepy - Building video /content/requested_presenter.mp4.
MoviePy - Writing audio in requested_presenterTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/requested_presenter.mp4



Moviepy - Done !
Moviepy - video ready /content/requested_presenter.mp4
Saved merged presenter clip to: /content/requested_presenter.mp4
